# SRA Custom Synapses (Vector DB & Calculator) Demo

In [1]:
import torch
import torch.nn.functional as F
import sys
import os
sys.path.append(os.path.abspath('../src'))
from sra_reference import SRAModel, VectorDBSynapse, CalculatorSynapse


## 1. SRAモデルの初期化

In [2]:
# SRAモデルの初期化（通常のニューラルシナプスを4つ持つ）
vocab_size = 100
dim = 32
layers = 2
num_synapses = 4
k = 1
syn_hidden = 64

model = SRAModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"初期のシナプス数: {model.blocks[0].router.num_synapses}")


初期のシナプス数: 4


## 2. カスタムシナプスのHot-swap追加

In [3]:
# VectorDBSynapseを追加
def create_vector_db():
    return VectorDBSynapse(dim)

def create_vector_db_emb():
    # 特定のドメイン用のEmbeddingを割り当てる（ランダム）
    return torch.randn(dim)

model.add_custom_synapse(create_vector_db, create_vector_db_emb)
print(f"VectorDBSynapse追加後のシナプス数: {model.blocks[0].router.num_synapses}")

# CalculatorSynapseを追加
def create_calculator():
    return CalculatorSynapse(dim)

def create_calc_emb():
    return torch.randn(dim)

model.add_custom_synapse(create_calculator, create_calc_emb)
print(f"CalculatorSynapse追加後のシナプス数: {model.blocks[0].router.num_synapses}")


VectorDBSynapse追加後のシナプス数: 5
CalculatorSynapse追加後のシナプス数: 6


## 3. VectorDBシナプスへの知識登録とテスト

In [4]:
# VectorDBシナプスに知識を追加する
# 最初のブロックのVectorDBシナプス(インデックス4)を取得
db_synapse = model.blocks[0].synapses[4]

# 「パリの首都は？」に相当する特徴量ベクトル（モック）と、その答え（事実ベクトル）を登録
fact_key = torch.randn(dim)
fact_value = torch.ones(dim) * 99  # 特徴的な値

db_synapse.add_knowledge(fact_key, fact_value)
print("事実データをVectorDBに登録しました。")

# 入力テスト: 事実キーと完全に一致する特徴量を入れた場合
x_in = fact_key.unsqueeze(0).unsqueeze(0)  # (1, 1, D)
out = db_synapse(x_in)
print(f"VectorDBの出力: {out[0, 0, :5]} ... (期待値: 99)")


事実データをVectorDBに登録しました。
VectorDBの出力: tensor([99., 99., 99., 99., 99.]) ... (期待値: 99)


## 4. ルーティングの確認

In [5]:
# モデル全体を通したルーティングのテスト
x = torch.randint(0, vocab_size, (1, 10))
y_in = torch.randint(0, vocab_size, (1, 5))

# 推論実行
out, router_logits, syn_outputs = model(x, y_in)

# RouterのLogitsを確認（各トークンがどのシナプスにルーティングされたか）
print(f"Router Logits (Block 0): shape {router_logits[0].shape}")
probs = F.softmax(router_logits[0], dim=-1)
print(f"各シナプスへのルーティング確率 (最初のトークン): \n{probs[0, 0, :]}")


Router Logits (Block 0): shape torch.Size([1, 15, 6])
各シナプスへのルーティング確率 (最初のトークン): 
tensor([0.0324, 0.4639, 0.1854, 0.0339, 0.0815, 0.2030],
       grad_fn=<SliceBackward0>)
